# 01 · População

Quantas pessoas vivem onde, e o quão desigual é essa distribuição.

**Fonte:** agregado 6579 (população residente estimada) e 4714 (Censo 2022), via API de Agregados v3 do IBGE.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import pandas as pd
from ibge_analytics.utils import io
from ibge_analytics.viz import charts, maps
from ibge_analytics.viz.theme import formatar_compacto, formatar_numero

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
painel = io.carregar("painel_municipios")
pop_ufs = io.carregar("populacao_ufs")
ano = int(painel["ano_populacao"].iloc[0])
print(f"{len(painel):,} municípios · estimativa {ano}")
print(f"População do Brasil: {painel['populacao_atual'].sum():,.0f}")

5,571 municípios · estimativa 2025
População do Brasil: 213,421,037


## Os maiores municípios

In [3]:
from ibge_analytics.analysis import populacao

ranking = populacao.ranking_municipios(painel, n=15)
ranking["rotulo"] = ranking["municipio_nome"] + " (" + ranking["uf_sigla"] + ")"
charts.barras_ranking(ranking, x="populacao_atual", y="rotulo",
                      rotulo_valor="População",
                      titulo=f"Municípios mais populosos ({ano})")

## Porte municipal

O contraste central da rede urbana brasileira: a maior parte dos municípios é pequena, e a maior parte da população não vive neles.

In [4]:
porte = populacao.distribuicao_por_porte(painel)
display(porte)
charts.barras_agrupadas_comparacao(
    porte, categoria="porte",
    series={"pct_municipios": "% dos municípios", "pct_populacao": "% da população"},
    titulo="Muitos municípios pequenos, pouca gente neles")

,porte,n_municipios,populacao,pct_municipios,pct_populacao
0,Até 5 mil,1285,"4,327,364.00",23.07,2.03
1,5 a 10 mil,1184,"8,477,694.00",21.25,3.97
2,10 a 20 mil,1350,"19,188,780.00",24.23,8.99
3,20 a 50 mil,1070,"32,995,428.00",19.21,15.46
4,50 a 100 mil,344,"24,029,096.00",6.17,11.26
5,100 a 500 mil,290,"58,543,655.00",5.21,27.43
6,Mais de 500 mil,48,"65,859,020.00",0.86,30.86


## Concentração

In [5]:
metricas = populacao.metricas_concentracao(painel)
print(f"Gini da população municipal: {metricas['gini']:.3f}")
print(f"Os 10% maiores concentram {metricas['share_top_10pct']:.1f}% da população")
print(f"Os 100 maiores concentram {metricas['share_top_100']:.1f}% da população")

charts.lorenz(populacao.curva_lorenz(painel),
              titulo="Curva de Lorenz da população municipal")

Gini da população municipal: 0.738
Os 10% maiores concentram 66.3% da população
Os 100 maiores concentram 40.1% da população


## Trajetória das regiões

Séries indexadas a 100 no primeiro ano: é o que permite comparar regiões de tamanhos muito diferentes num eixo só, sem recorrer a um segundo eixo y.

In [6]:
por_regiao = pop_ufs.groupby(["ano", "regiao_nome"], observed=True,
                             as_index=False)["populacao"].sum()
base = por_regiao[por_regiao["ano"] == por_regiao["ano"].min()]\
    .set_index("regiao_nome")["populacao"]
por_regiao["indice"] = por_regiao["populacao"] / por_regiao["regiao_nome"].map(base) * 100

charts.linha_temporal(por_regiao, x="ano", y="indice", cor="regiao_nome",
                      titulo="População por região (base 100)")

> A série de estimativas não cobre 2007, 2010, 2022 e 2023 — anos de Censo/Contagem ou de estimativa suspensa para revisão. As linhas ligam os anos publicados.